# Imports

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Input Data

In [6]:
budget = 100000
risk = 23
expected_return = 15
stocks = ["AAPL","^GSPC"]
days = 365

# VaR 

In [7]:

import numpy as np

def VaR(ticker, p=0.95):
    try:
        import yfinance as yf
    except ImportError:
        import os
        os.system("pip install yfinance")
        import yfinance as yf

    try:
        from scipy.stats import norm
    except ImportError:
        import os
        os.system("pip install scipy")
        from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker, progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")


    returns = (data['Open']-data['Close'])/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    sigma = returns.std()
    Z_p = norm.ppf(1 - p)
    VaR = (Z_p * sigma) - E
    return VaR 

# RoI

In [22]:

import numpy as np

def RoI(ticker, p=0.95,days=365):
    try:
        import yfinance as yf
    except ImportError:
        import os
        os.system("pip install yfinance")
        import yfinance as yf

    try:
        from scipy.stats import norm
    except ImportError:
        import os
        os.system("pip install scipy")
        from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker,start="1900-01-01" ,progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'].shift(days))/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    return E 

# Optimization model

In [26]:
try:
    import pulp as pl
except ImportError:
    import os
    os.system("pip install pulp")
    import pulp as pl

from IPython.display import clear_output

risk = risk/100
expected_return = expected_return/100

# define problem
z = pl.LpProblem("Portfolio_Optimization", pl.LpMinimize)

# define decision variables
inv = pl.LpVariable.dicts("Stocks", stocks, lowBound=0, cat='Continuous')

#define continuity variable
total_spent = pl.LpVariable("Total_Investment", lowBound=0, cat='Continuous')

# risk slack variable
s1 = pl.LpVariable("Risk_Slack", lowBound=0, cat='Continuous')
s2 = pl.LpVariable("Return_Slack", lowBound=0, cat='Continuous')

VaRs = {ticker: VaR(ticker) for ticker in stocks}
RoIs = {ticker: RoI(ticker, days=days) for ticker in stocks}

# define objective function
z += s1 + s2, "Minimize_Slack_Variables"

# continuity constraints
z+= pl.lpSum([inv[ticker] for ticker in inv]) == total_spent
z+= total_spent <= budget
z+= pl.lpSum([inv[ticker] for ticker in inv]) >= 1
# define constraints
z += pl.lpSum([inv[ticker] for ticker in inv]) <= budget, "Budget_Constraint"
z += pl.lpSum([inv[ticker] * VaRs[ticker] for ticker in inv]) == total_spent * risk + s1, "Risk_Constraint"
z += pl.lpSum([inv[ticker] * RoIs[ticker] for ticker in inv]) == total_spent * expected_return - s2, "Return_Constraint"



# solve problem
z.solve()
clear_output()

print("-"*200)

# print inputs
print("Inputs:")
print("#"*20)
print("Budget:", budget)
print("Stocks:", stocks)
print("Risk (VaR):", risk)
print("Expected Return:", expected_return)



# print results
for v in z.variables():
    print(f"{v.name}: {v.varValue}")
print(f"Objective value: {pl.value(z.objective)}")

print("\n"*3)
print("Total Expected Return:", sum([inv[ticker].varValue * RoIs[ticker] for ticker in inv]))
print("Total Risk (VaR):", sum([inv[ticker].varValue * VaRs[ticker] for ticker in inv]))

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Inputs:
####################
Budget: 100000
Stocks: ['AAPL', '^GSPC']
Risk (VaR): 2.2999999999999996e-29
Expected Return: 1.5e-29
Return_Slack: 0.0
Risk_Slack: 0.0
Stocks_AAPL: -0.94643024
Stocks_^GSPC: 1.9464302
Total_Investment: 1.0
Objective value: 0.0




Total Expected Return: -1.7908038107972146e-09
Total Risk (VaR): -0.0056815233260800735
